### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [2]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [3]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [4]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [5]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [6]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [7]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [8]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [9]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some pros of using AutoGen in AI Agent projects:

1. **Simplification of Development**: AutoGen provides a framework that simplifies the orchestration of multiple AI agents, allowing developers to focus on functionality rather than complex inter-agent protocols.

2. **Enhanced Collaboration**: AutoGen facilitates seamless communication between AI agents, enabling them to converse and collaborate effectively, which can lead to more refined outputs and quicker iterations.

3. **Efficiency in Workflows**: The framework helps to optimize workflows involving LLMs (Large Language Models), improving overall efficiency in developing AI applications.

4. **Flexibility and Customization**: AutoGen allows for customizable agent configurations, enabling developers to tailor interactions and capabilities based on specific project requirements.

5. **Open-Source**: Being an open-source framework, AutoGen promotes community collaboration and continuous improvement, which can be beneficial for long-term project sustainability and support.

6. **Integration with Cloud Services**: AutoGen can easily integrate with Azure, enhancing scalability and the potential for broader application deployment.

7. **Reduced Coordination Complexity**: By utilizing natural language for handoffs between agents, AutoGen reduces the technical overhead involved in coordinating multiple agents.

These advantages make AutoGen a compelling choice for businesses looking to implement effective AI solutions with multiple interacting agents.

TERMINATE

## Cons of AutoGen:
Here are some potential cons of using AutoGen for your AI Agent project:

1. **Inconsistent Outputs**: AutoGen may produce varying results, leading to reliability issues in applications where consistency is critical.

2. **Difficult Debugging**: The complexity of AutoGen can make it challenging to trace errors and bugs, complicating the development and maintenance process.

3. **Rising Costs**: Depending on usage and scaling, operational costs can increase significantly, particularly if extensive infrastructure is required.

4. **Fragile Behavior**: The behavior of agents created with AutoGen may be fragile, meaning they could fail in unexpected ways or produce unanticipated outcomes when conditions change.

5. **Challenging Documentation**: Users report that the documentation for AutoGen is difficult to read, lacking sufficient examples which can complicate the onboarding process for new developers.

6. **Over-Engineering**: The framework can lead to overly complex solutions that may not be necessary for all use cases, making projects more cumbersome to manage.

7. **Customization Requirements**: AutoGen may need significant customization to fit specific use cases, which can extend the development timeline and require additional resources.

8. **Infrastructure Management**: Users often need to handle their own hosting and infrastructure management, which can add to the overhead and complexity of deployment.

These factors should be considered when deciding whether AutoGen is the right fit for your new AI Agent project. 

TERMINATE



## Decision:

Based on the analysis of the pros and cons of AutoGen, I recommend moving forward with its use for the project. The significant advantages in simplifying development, enhancing collaboration, optimizing workflows, and providing flexibility and open-source benefits outweigh the potential challenges such as inconsistency and complex debugging. Given the project's needs for effective interaction between AI agents and the capability to scale with cloud integration, the benefits align closely with our goals. 

However, it will be essential to anticipate the challenges and invest in strategies for robust testing and error handling to mitigate the risks identified.

TERMINATE

In [10]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [11]:
await host.stop()